In [1]:
import requests
from bs4 import BeautifulSoup
import re
import numpy as np
import pandas as pd
import time
import random

In [2]:
session = requests.Session()
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:106.0) Gecko/20100101 Firefox/106.0',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,/;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'DNT': '1',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-User': '?1',
}

session.headers.update(headers)

In [3]:
links = [
    "https://www.99acres.com/search/property/rent/residential-all/hyderabad?city=7&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/bangalore?city=20&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/chennai?city=32&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/mumbai?city=12&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/delhi?city=4&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/kolkata?city=9&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/ahmedabad?city=37&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/surat?city=38&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/jaipur?city=39&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/lucknow?city=49&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/kanpur?city=50&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/patna?city=118&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/chandigarh?city=14&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/mysore?city=157&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/coimbatore?city=158&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/kochi?city=173&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/visakhapatnam?city=83&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/warangal?city=247&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/tirupati?city=260&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/kurnool?city=261&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
    "https://www.99acres.com/search/property/rent/residential-all/nellore?city=262&bedroom_num=1%2C2%2C3%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
]

In [ ]:
Seller_Type=[]
name=[]
types=[]
Area=[]
City=[]
Measure=[]
price = []
Deposit=[]
Dealer_Name = []
for idx in range(len(links)):
    for j in range(1,10):
        url=links[idx] + str(j)
        page=None
        page=requests.get(url,headers=headers)
        print(page)
        for attempt in range(3):
            
            try:
                page = session.get(url, timeout=10)
                
                if page.status_code == 200:
                    print(f"200 for {url}")
                    soup = BeautifulSoup(page.text)
                    break
                
                else:
                    print(f"Attempt {attempt+1} failed:", page.status_code)
                    time.sleep(random.uniform(5,10))
            
            except requests.exceptions.RequestException as e:
                print("Connection Error:", e)
                time.sleep(random.uniform(5,10))
        
        else:
            print("Skipping:", url)
        na=soup.find_all("div",class_="tupleNew__locationName ellipsis")
        for i in na:
            name.append(i.text)
        var = soup.find_all("h2",class_="tupleNew__propType")
        for i in var:
            types.append(i.text.split(" ")[0])
        for i in var:
            if i.text:
                a=re.search("in\s+(.+?)(,| )",i.text)
                Area.append(a.group(1))
            else:
                Area.append("NA")
        for i in var:
            a=re.findall(r"in\s+\w+,?\s(.+)$",i.text)
            if a:
                City.append(a[0])
            else:
                City.append("NA")
                
        meas =  soup.find_all("span",class_="tupleNew__area1Type")
        for i in range(0,len(meas),2):
            Measure.append(meas[i].text)
        pric = soup.find_all("div",class_="tupleNew__priceValWrap")
        for i in pric:
            price.append(i.text.split("/")[0].strip())
        depo = soup.find_all("div",class_="tupleNew__priceAndPerSqftWrap")
        for i in depo:
            p=re.findall("month\+\s+\w+\s(.+)", i.text)
            if p:
                Deposit.append(p[0])
            else:
                Deposit.append("NA")
        deal = soup.find_all("div",class_="tupleNew__tupleWrapTopaz tupleNew__platHTopaz")

In [196]:
a=[
name,
types,
Area,
City,
Measure,
price,
Deposit,
]
for i in a:
    print(len(i))


1983
1983
1983
1983
1983
1983
1983


In [149]:
for i in range(1,4):
    print(i)

1
2
3


In [197]:
df=pd.DataFrame({"Name":name,"Type of Appartment(BHK)":types,"Measurements":Measure,"Price":price,"Deposit":Deposit,"Area":Area,"City":City,})

In [198]:
df

,Name,Type of Appartment(BHK),Measurements,Price,Deposit,Area,City
0,3C Lotus Panache,2,"1,220 sqft","₹27,000","₹54,000",Sector,"110, Noida"
1,Antriksh Nature,3,"1,695 sqft","₹40,000","₹4,000",Sector,"52, Noida"
2,Amrapali HeartBeat City,3,"1,735 sqft","₹52,000",2 months rent,Sector,"107, Noida"
3,Panchsheel Pratishtha,3,"2,050 sqft","₹42,500",2 months rent,Sector,"75, Noida"
4,Kalpataru Vista,3,"3,000 sqft",₹1.1 Lac,2 months rent,Sector,"128, Noida"
...,...,...,...,...,...,...,...
1978,Ram Sharman Colony,2,750 sqft,"₹12,000",NA,Bhadroya,Pathankot
1979,"Malikpur, Pathankot",1,450 sqft,"₹4,500",NA,Malikpur,Pathankot
1980,"Lamini, Pathankot",2,"1,755 sqft","₹15,000","₹30,000",Lamini,Pathankot
1981,"shivaji nagar, Pathankot",2,"2,000 sqft","₹10,000",NA,shivaji,"nagar, Pathankot"


In [202]:
df.drop_duplicates(inplace=True)

In [205]:
df.reset_index(inplace=True)

In [206]:
df

,index,Name,Type of Appartment(BHK),Measurements,Price,Deposit,Area,City
0,0,3C Lotus Panache,2,"1,220 sqft","₹27,000","₹54,000",Sector,"110, Noida"
1,1,Antriksh Nature,3,"1,695 sqft","₹40,000","₹4,000",Sector,"52, Noida"
2,2,Amrapali HeartBeat City,3,"1,735 sqft","₹52,000",2 months rent,Sector,"107, Noida"
3,3,Panchsheel Pratishtha,3,"2,050 sqft","₹42,500",2 months rent,Sector,"75, Noida"
4,4,Kalpataru Vista,3,"3,000 sqft",₹1.1 Lac,2 months rent,Sector,"128, Noida"
...,...,...,...,...,...,...,...,...
272,1978,Ram Sharman Colony,2,750 sqft,"₹12,000",NA,Bhadroya,Pathankot
273,1979,"Malikpur, Pathankot",1,450 sqft,"₹4,500",NA,Malikpur,Pathankot
274,1980,"Lamini, Pathankot",2,"1,755 sqft","₹15,000","₹30,000",Lamini,Pathankot
275,1981,"shivaji nagar, Pathankot",2,"2,000 sqft","₹10,000",NA,shivaji,"nagar, Pathankot"


In [207]:
pint=[]
for i in df["Price"]:
    if "L" in i:
        pint.append(float(i.split(" ")[0].replace("₹","")))
    else:
        pint.append(int(i.split("₹")[1].replace(",","")))

In [208]:
len(pint)

277

In [209]:
Measure=[]
for i in df["Measurements"]:
    Measure.append(int(i.split(" ")[0].replace(",","")))

In [210]:
prsq=[]
for i in range(len(pint)):
    prsq.append(round(pint[i]/Measure[i],2))
    

In [211]:
df["Price/sqft"]=prsq

In [230]:
df.drop("index",axis=1,inplace=True)

In [129]:
df.to_excel("Real_Estate_Rent_Dataframe.xlsx",index=False)

In [130]:
links=["https://www.99acres.com/search/property/rent/residential-all/pune?city=19&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/haryana?city=234&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/guntur?city=54&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/vijayawada?city=61&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/ranchi?city=117&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/raipur?city=75&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/gujarat?city=225&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/jharkhand?city=237&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/srinagar?city=112&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/nagpur?city=150&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/tripura?city=248&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/madurai?city=188&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/ujjain?city=144&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/kanchipuram?city=444&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/bodhgaya-gaya?city=499&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&locality=19107&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/indore?city=142&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C2%2C4%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/thane?city=219&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=R&area_unit=1&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/bhopal?city=140&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/agra?city=197&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/nashik?city=151&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/rent/residential-all/rajkot?city=94&bedroom_num=1%2C2%2C3%2C4%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=R&area_unit=1&budget_min=0&res_com=R&isPreLeased=N"]

In [137]:
cities_done=[]
for i in links:
    cities_done.append((re.findall("-all/(.+)\?",i)[0]))

In [140]:
print(cities_done)

['pune', 'haryana', 'guntur', 'vijayawada', 'ranchi', 'raipur', 'gujarat', 'jharkhand', 'srinagar', 'nagpur', 'tripura', 'madurai', 'ujjain', 'kanchipuram', 'bodhgaya-gaya', 'indore', 'thane', 'bhopal', 'agra', 'nashik', 'rajkot']


In [139]:
len(cities_done)

21

In [182]:
df1=pd.read_excel("C:\\Users\\D.Sagar\\OneDrive\\Desktop\\Innomatics_Folder\\Projects\\Real_Estate_Rent_Dataframe.xlsx")

In [183]:
df1

,Name,Type of Appartment(BHK),Measurements,Price,Deposit,Area,City,Price/sqft
0,Life Republic I Tower,2,650 sqft,"₹22,000","₹50,000",Hinjewadi,Pune,33.85
1,Yahavi Vanaha Bavdhan,2,"1,100 sqft","₹21,500",2 months rent,Patil,"Nagar, Bavdhan",19.55
2,Rama Melange Residences,3,963 sqft,"₹35,000","₹70,000",Hinjewadi,Pune,36.34
3,Godrej Greens,3,780 sqft,"₹23,000","₹25,000",Undri,Pune,29.49
4,g building florentine housing society pune 1,2,"1,210 sqft","₹60,000",NaN,Ghorpadi,Pune,49.59
...,...,...,...,...,...,...,...,...
543,At Amin Marg,4,"1,800 sqft","₹35,000","₹60,000",Sirvar,"Park, Rajkot",19.44
544,RK Prime,1,323 sqft,"₹28,000","₹56,000",Nana,"Mava, Rajkot",86.69
545,Shree Hari Darshan,1,400 sqft,"₹9,500","₹9,500",Raiya,"Road, Rajkot",23.75
546,Sopan Heights,3,743 sqft,"₹20,000","₹20,000",Raiya,Rajkot,26.92


In [184]:
df_1=pd.concat([df,df1], axis=0) 

In [186]:
df_1.duplicated().sum()

np.int64(411)

In [187]:
df_1.drop_duplicates(inplace=True)

In [189]:
df_1.reset_index()

,index,Name,Type of Appartment(BHK),Measurements,Price,Deposit,Area,City,Price/sqft
0,0,Life Republic I Tower,2,650 sqft,"₹22,000","₹50,000",Hinjewadi,Pune,33.85
1,1,Shapoorji Bavdhan,1,700 sqft,"₹16,000",2 months rent,Bavdhan,Pune,22.86
2,2,Rama Melange Residences,3,963 sqft,"₹35,000","₹70,000",Hinjewadi,Pune,36.34
3,3,Godrej Greens,3,780 sqft,"₹23,000","₹25,000",Undri,Pune,29.49
4,4,g building florentine housing society pune 1,2,"1,210 sqft","₹60,000",NA,Ghorpadi,Pune,49.59
...,...,...,...,...,...,...,...,...,...
1531,528,"Udhyognagar, Rajkot",2,799 sqft,"₹18,000",NaN,Udhyognagar,Rajkot,22.53
1532,529,"Udhyognagar, Rajkot",2,800 sqft,"₹18,000",NaN,Udhyognagar,Rajkot,22.50
1533,533,"Gopal Nagar, Rajkot",3,720 sqft,"₹11,500",NaN,Gopal,"Nagar, Rajkot",15.97
1534,535,Rail nagar,3,657 sqft,"₹15,000",NaN,Radhika,"residency 2, Rajkot",22.83


In [190]:
df_1.to_excel("Real_Estate_Rent_Dataframe_1.xlsx",index=False)

In [216]:
df_1.drop_duplicates(inplace=True)

In [217]:
df_1.reset_index(inplace=True)

In [225]:
df_1.drop("index",axis=1,inplace=True)

In [226]:
df_1

,Name,Type of Appartment(BHK),Measurements,Price,Deposit,Area,City,Price/sqft
0,Life Republic I Tower,2,650 sqft,"₹22,000","₹50,000",Hinjewadi,Pune,33.85
1,Shapoorji Bavdhan,1,700 sqft,"₹16,000",2 months rent,Bavdhan,Pune,22.86
2,Rama Melange Residences,3,963 sqft,"₹35,000","₹70,000",Hinjewadi,Pune,36.34
3,Godrej Greens,3,780 sqft,"₹23,000","₹25,000",Undri,Pune,29.49
4,g building florentine housing society pune 1,2,"1,210 sqft","₹60,000",NA,Ghorpadi,Pune,49.59
...,...,...,...,...,...,...,...,...
1531,"Udhyognagar, Rajkot",2,799 sqft,"₹18,000",NaN,Udhyognagar,Rajkot,22.53
1532,"Udhyognagar, Rajkot",2,800 sqft,"₹18,000",NaN,Udhyognagar,Rajkot,22.50
1533,"Gopal Nagar, Rajkot",3,720 sqft,"₹11,500",NaN,Gopal,"Nagar, Rajkot",15.97
1534,Rail nagar,3,657 sqft,"₹15,000",NaN,Radhika,"residency 2, Rajkot",22.83


In [231]:
DF_2=pd.concat([df,df_1], axis=0) 

In [233]:
DF_2.reset_index(inplace=True)

In [234]:
DF_2

,index,Name,Type of Appartment(BHK),Measurements,Price,Deposit,Area,City,Price/sqft
0,0,3C Lotus Panache,2,"1,220 sqft","₹27,000","₹54,000",Sector,"110, Noida",22.13
1,1,Antriksh Nature,3,"1,695 sqft","₹40,000","₹4,000",Sector,"52, Noida",23.60
2,2,Amrapali HeartBeat City,3,"1,735 sqft","₹52,000",2 months rent,Sector,"107, Noida",29.97
3,3,Panchsheel Pratishtha,3,"2,050 sqft","₹42,500",2 months rent,Sector,"75, Noida",20.73
4,4,Kalpataru Vista,3,"3,000 sqft",₹1.1 Lac,2 months rent,Sector,"128, Noida",0.00
...,...,...,...,...,...,...,...,...,...
1808,1531,"Udhyognagar, Rajkot",2,799 sqft,"₹18,000",NaN,Udhyognagar,Rajkot,22.53
1809,1532,"Udhyognagar, Rajkot",2,800 sqft,"₹18,000",NaN,Udhyognagar,Rajkot,22.50
1810,1533,"Gopal Nagar, Rajkot",3,720 sqft,"₹11,500",NaN,Gopal,"Nagar, Rajkot",15.97
1811,1534,Rail nagar,3,657 sqft,"₹15,000",NaN,Radhika,"residency 2, Rajkot",22.83


In [235]:
DF_2.to_excel("Real_Estate_Rent_Dataframe_2.xlsx",index=False)